In [1]:
import argparse
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# -----------------------------
# Basic utilities
# -----------------------------

def normalize_column_name(name: str) -> str:
    """
    Convert column names to lowercase snake_case.

    Example:
      "Flow Bytes/s" -> "flow_bytes_s"
      "Destination Port" -> "destination_port"
      "Init_Win_bytes_forward" -> "init_win_bytes_forward"
    """
    name = name.strip().lower()
    name = name.replace("/", "_")
    name = name.replace("-", "_")
    name = name.replace(" ", "_")
    name = re.sub(r"[^a-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    return name


def read_csv_safely(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{name} file not found: {path}")

    df = pd.read_csv(path, low_memory=False)
    # df.columns = [normalize_column_name(c) for c in df.columns]
    # 去除掉flow_id为0的数据
    df = df[df['flow_id'] != 0]


    print(f"[OK] Loaded {name}: {path}")
    print(f"     shape = {df.shape}")
    return df


def ensure_output_dirs(out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    plot_dir = out_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    return plot_dir


def convert_numeric_columns(df, exclude_cols=None):
    result = df.copy()
    if exclude_cols is None:
        exclude_cols = set()
    for col in result.columns:
        if col in exclude_cols:
            continue
        converted = pd.to_numeric(result[col], errors='coerce')
        # 将转换失败（变为 NaN）的值重新填回原始值
        mask = converted.isna()
        converted[mask] = result[col][mask]
        result[col] = converted
    return result


def numeric_columns(df: pd.DataFrame):
    return df.select_dtypes(include=[np.number]).columns.tolist()


def save_missing_report(df: pd.DataFrame, path: Path):
    rows = []
    n = len(df)

    for col in df.columns:
        missing = df[col].isna().sum()
        rows.append({
            "column": col,
            "missing_count": int(missing),
            "missing_ratio": float(missing / n) if n > 0 else 0.0,
            "dtype": str(df[col].dtype),
        })

    report = pd.DataFrame(rows).sort_values(
        ["missing_count", "column"],
        ascending=[False, True]
    )
    report.to_csv(path, index=False)


def save_numeric_summary(df: pd.DataFrame, path: Path):
    num_cols = numeric_columns(df)

    if not num_cols:
        pd.DataFrame().to_csv(path, index=False)
        return

    summary = df[num_cols].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ).T

    summary["missing_count"] = df[num_cols].isna().sum()
    summary["inf_count"] = np.isinf(df[num_cols].replace([np.inf, -np.inf], np.nan)).sum()
    summary["zero_count"] = (df[num_cols] == 0).sum()
    summary["negative_count"] = (df[num_cols] < 0).sum()

    summary.to_csv(path)


def safe_log1p(series: pd.Series) -> pd.Series:
    """
    log(1 + x), with negative values clipped to 0.
    """
    x = pd.to_numeric(series, errors="coerce").fillna(0)
    x = x.clip(lower=0)
    return np.log1p(x)


def plot_hist_log1p(series: pd.Series, path: Path, title: str, xlabel: str):
    x = pd.to_numeric(series, errors="coerce").dropna()
    x = x[np.isfinite(x)]
    x = x.clip(lower=0)

    if len(x) == 0:
        return

    plt.figure(figsize=(8, 5))
    plt.hist(np.log1p(x), bins=80)
    plt.title(title)
    plt.xlabel(f"log1p({xlabel})")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def plot_bar_counts(series: pd.Series, path: Path, title: str, xlabel: str):
    counts = series.value_counts(dropna=False).sort_index()

    if len(counts) == 0:
        return

    plt.figure(figsize=(8, 5))
    counts.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


In [3]:
# -----------------------------
# Flow-packet consistency checks
# -----------------------------

def build_packet_flow_summary(packets: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate packet CSV by flow_id.

    Required packet columns:
      flow_id
      direction
      packet_label
      flow_iat_us
      direction_iat_us
    """
    if "flow_id" not in packets.columns:
        return pd.DataFrame()

    p = packets.copy()

    if "direction" not in p.columns:
        p["direction"] = np.nan
    if "packet_label" not in p.columns:
        p["packet_label"] = 0
    if "flow_iat_us" not in p.columns:
        p["flow_iat_us"] = 0
    if "direction_iat_us" not in p.columns:
        p["direction_iat_us"] = 0
    if "payload_len" not in p.columns:
        p["payload_len"] = 0

    p["direction"] = pd.to_numeric(p["direction"], errors="coerce")
    p["packet_label"] = pd.to_numeric(p["packet_label"], errors="coerce").fillna(0)
    p["flow_iat_us"] = pd.to_numeric(p["flow_iat_us"], errors="coerce").fillna(0)
    p["direction_iat_us"] = pd.to_numeric(p["direction_iat_us"], errors="coerce").fillna(0)
    p["payload_len"] = pd.to_numeric(p["payload_len"], errors="coerce").fillna(0)

    grouped = p.groupby("flow_id", dropna=False).agg(
        packet_rows=("flow_id", "size"),
        packet_fwd_count=("direction", lambda s: int((s == 0).sum())),
        packet_bwd_count=("direction", lambda s: int((s == 1).sum())),
        packet_unknown_direction_count=("direction", lambda s: int((~s.isin([0, 1])).sum())),
        any_packet_label=("packet_label", lambda s: int((s > 0).any())),
        alert_packet_count=("packet_label", lambda s: int((s > 0).sum())),
        packet_payload_sum=("payload_len", "sum"),
        flow_iat_mean_from_packets=("flow_iat_us", "mean"),
        flow_iat_max_from_packets=("flow_iat_us", "max"),
        direction_iat_mean_from_packets=("direction_iat_us", "mean"),
        direction_iat_max_from_packets=("direction_iat_us", "max"),
    ).reset_index()

    return grouped


def check_flow_packet_consistency(flows: pd.DataFrame, packets: pd.DataFrame, out_dir: Path):
    if "flow_id" not in flows.columns or "flow_id" not in packets.columns:
        return pd.DataFrame(), pd.DataFrame()

    packet_summary = build_packet_flow_summary(packets)

    keep_cols = ["flow_id"]

    for col in [
        "total_fwd_packets",
        "total_backward_packets",
        "label",
        "flow_duration",
        "flow_iat_mean",
        "flow_iat_max",
    ]:
        if col in flows.columns:
            keep_cols.append(col)

    f = flows[keep_cols].copy()
    merged = f.merge(packet_summary, on="flow_id", how="outer", indicator=True)

    if "total_fwd_packets" in merged.columns:
        merged["fwd_packet_diff"] = (
            pd.to_numeric(merged["total_fwd_packets"], errors="coerce")
            - pd.to_numeric(merged["packet_fwd_count"], errors="coerce")
        )

    if "total_backward_packets" in merged.columns:
        merged["bwd_packet_diff"] = (
            pd.to_numeric(merged["total_backward_packets"], errors="coerce")
            - pd.to_numeric(merged["packet_bwd_count"], errors="coerce")
        )

    merged.to_csv(out_dir / "flow_packet_consistency.csv", index=False)

    # Label consistency:
    # if any packet_label == 1, flow label should be 1.
    label_consistency = pd.DataFrame()

    if "label" in merged.columns and "any_packet_label" in merged.columns:
        label_consistency = merged[[
            "flow_id",
            "label",
            "any_packet_label",
            "alert_packet_count",
            "_merge"
        ]].copy()

        label_consistency["label"] = pd.to_numeric(
            label_consistency["label"], errors="coerce"
        ).fillna(-1).astype(int)

        label_consistency["label_mismatch"] = (
            label_consistency["label"] != label_consistency["any_packet_label"]
        )

        label_consistency.to_csv(out_dir / "label_consistency.csv", index=False)

    return merged, label_consistency



In [4]:
# -----------------------------
# Feature preparation
# -----------------------------

def create_enriched_packets(packets: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    """
    Add log-IAT features for Stage-I time-aware Transformer.
    record_type,packet_id,timestamp_us,ts_sec,ts_usec,flow_id,direction,ip_version,src_ip,dst_ip,src_port,dst_port,protocol,pkt_len,ip_len,payload_len,ttl_or_hop_limit,tcp_seq,tcp_ack,tcp_flags,tcp_window,tcp_header_len,udp_len,icmp_type,icmp_code,l7_proto,flow_iat_us,direction_iat_us,packet_label

    """
    p = packets.copy()

    if "flow_iat_us" in p.columns:
        p["log_flow_iat"] = safe_log1p(p["flow_iat_us"])
    else:
        p["log_flow_iat"] = 0.0

    if "direction_iat_us" in p.columns:
        p["log_direction_iat"] = safe_log1p(p["direction_iat_us"])
    else:
        p["log_direction_iat"] = 0.0

    # Per-flow packet order for positional encoding.
    if "flow_id" in p.columns and "timestamp_us" in p.columns:
        # 方法 1：在排序前检查列是否存在（精准且高效）
        # 这就是你报错的那一行
        # p = p.sort_values(["flow_id", "timestamp_us", "packet_id"], errors="ignore")

        # 替换方案：只对存在的列进行排序
        sort_cols = ["flow_id", "timestamp_us", "packet_id"]
        existing_cols = [col for col in sort_cols if col in p.columns]
        if existing_cols:
            p = p.sort_values(existing_cols)
        p["packet_index_in_flow"] = p.groupby("flow_id").cumcount()
    elif "flow_id" in p.columns:
        p["packet_index_in_flow"] = p.groupby("flow_id").cumcount()
    else:
        p["packet_index_in_flow"] = np.arange(len(p))

    p.to_csv(out_dir / "stage1_packets_enriched.csv", index=False)
    return p


def create_flow_ml_features(flows: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    """
    Create a flow-level ML table.

    Drops identifiers and string columns by default.
    Keeps label as target.
    """
    f = flows.copy()

    drop_cols = [
        "flow_id",
        "source_ip",
        "destination_ip",
        "src_ip",
        "dst_ip",
        "record_type",
    ]

    existing_drop_cols = [c for c in drop_cols if c in f.columns]
    f = f.drop(columns=existing_drop_cols)

    # Convert any remaining non-label columns to numeric where possible.
    for col in f.columns:
        if col == "label":
            f[col] = pd.to_numeric(f[col], errors="coerce").fillna(0).astype(int)
        else:
            f[col] = pd.to_numeric(f[col], errors="coerce")

    # Replace inf with NaN, then fill NaN with 0 for baseline models.
    f = f.replace([np.inf, -np.inf], np.nan)
    f = f.fillna(0)

    f.to_csv(out_dir / "stage1_flows_ml_features.csv", index=False)
    return f


def compute_label_correlations(flow_ml: pd.DataFrame, out_dir: Path):
    if "label" not in flow_ml.columns:
        return pd.DataFrame()

    label = pd.to_numeric(flow_ml["label"], errors="coerce")

    if label.nunique(dropna=True) < 2:
        return pd.DataFrame()

    feature_cols = [c for c in flow_ml.columns if c != "label"]
    corrs = {}

    for col in feature_cols:
        x = pd.to_numeric(flow_ml[col], errors="coerce")
        if x.nunique(dropna=True) <= 1:
            continue

        corr = x.corr(label)
        if pd.notna(corr):
            corrs[col] = corr

    result = pd.DataFrame({
        "feature": list(corrs.keys()),
        "pearson_corr_with_label": list(corrs.values()),
    })

    if not result.empty:
        result["abs_corr"] = result["pearson_corr_with_label"].abs()
        result = result.sort_values("abs_corr", ascending=False)
        result.to_csv(out_dir / "top_label_correlations.csv", index=False)

    return result


In [5]:
# -----------------------------
# Main report
# -----------------------------

def write_markdown_report(
    packets: pd.DataFrame,
    flows: pd.DataFrame,
    enriched_packets: pd.DataFrame,
    flow_ml: pd.DataFrame,
    consistency: pd.DataFrame,
    label_consistency: pd.DataFrame,
    corrs: pd.DataFrame,
    out_dir: Path,
):
    report_path = out_dir / "report.md"

    lines = []
    lines.append("# Stage-I Suricata Output Analysis Report\n")

    lines.append("## 1. Dataset size\n")
    lines.append(f"- packets rows: `{len(packets):,}`")
    lines.append(f"- flows rows: `{len(flows):,}`")
    lines.append(f"- packet columns: `{packets.shape[1]}`")
    lines.append(f"- flow columns: `{flows.shape[1]}`")

    if "flow_id" in packets.columns:
        lines.append(f"- unique flow_id in packets: `{packets['flow_id'].nunique(dropna=True):,}`")
    if "flow_id" in flows.columns:
        lines.append(f"- unique flow_id in flows: `{flows['flow_id'].nunique(dropna=True):,}`")

    lines.append("\n## 2. Label distribution\n")

    if "packet_label" in packets.columns:
        packet_label_counts = packets["packet_label"].value_counts(dropna=False).sort_index()
        lines.append("### Packet labels\n")
        lines.append(packet_label_counts.to_markdown())

    if "label" in flows.columns:
        flow_label_counts = flows["label"].value_counts(dropna=False).sort_index()
        lines.append("\n### Flow labels\n")
        lines.append(flow_label_counts.to_markdown())

        if len(flows) > 0:
            attack_ratio = (pd.to_numeric(flows["label"], errors="coerce") == 1).mean()
            lines.append(f"\n- flow attack ratio: `{attack_ratio:.6f}`")

    lines.append("\n## 3. Protocol distribution\n")

    if "protocol" in flows.columns:
        lines.append("### Flow protocol counts\n")
        lines.append(flows["protocol"].value_counts(dropna=False).sort_index().to_markdown())

    if "protocol" in packets.columns:
        lines.append("\n### Packet protocol counts\n")
        lines.append(packets["protocol"].value_counts(dropna=False).sort_index().to_markdown())

    lines.append("\n## 4. IAT summary\n")

    for col in ["flow_iat_us", "direction_iat_us"]:
        if col in packets.columns:
            s = pd.to_numeric(packets[col], errors="coerce")
            lines.append(f"\n### `{col}`\n")
            lines.append(s.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_markdown())

    lines.append("\n## 5. Flow-packet consistency\n")

    if not consistency.empty:
        lines.append(f"- consistency rows: `{len(consistency):,}`")
        if "_merge" in consistency.columns:
            lines.append("\n### merge status\n")
            lines.append(consistency["_merge"].value_counts().to_markdown())

        if "fwd_packet_diff" in consistency.columns:
            mismatch_fwd = (consistency["fwd_packet_diff"].fillna(0) != 0).sum()
            lines.append(f"- fwd packet count mismatches: `{mismatch_fwd:,}`")

        if "bwd_packet_diff" in consistency.columns:
            mismatch_bwd = (consistency["bwd_packet_diff"].fillna(0) != 0).sum()
            lines.append(f"- bwd packet count mismatches: `{mismatch_bwd:,}`")

    if not label_consistency.empty and "label_mismatch" in label_consistency.columns:
        mismatch_label = label_consistency["label_mismatch"].sum()
        lines.append(f"- label mismatches: `{mismatch_label:,}`")

    lines.append("\n## 6. Top correlations with flow label\n")

    if corrs is not None and not corrs.empty:
        lines.append(corrs.head(25)[["feature", "pearson_corr_with_label"]].to_markdown(index=False))
    else:
        lines.append("No correlation report generated. Possible reasons: no `label` column or only one class.")

    lines.append("\n## 7. Generated files\n")
    lines.append("- `numeric_summary_packets.csv`")
    lines.append("- `numeric_summary_flows.csv`")
    lines.append("- `missing_values_packets.csv`")
    lines.append("- `missing_values_flows.csv`")
    lines.append("- `flow_packet_consistency.csv`")
    lines.append("- `label_consistency.csv`")
    lines.append("- `top_label_correlations.csv`")
    lines.append("- `stage1_packets_enriched.csv`")
    lines.append("- `stage1_flows_ml_features.csv`")
    lines.append("- `plots/*.png`")

    report_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"[OK] Wrote report: {report_path}")


In [6]:
# -----------------------------
# Plot generation
# -----------------------------

def generate_plots(packets: pd.DataFrame, flows: pd.DataFrame, plot_dir: Path):
    if "label" in flows.columns:
        plot_bar_counts(flows["label"], plot_dir / "flow_label_distribution.png",
                        "Flow label distribution", "label")

    if "packet_label" in packets.columns:
        plot_bar_counts(packets["packet_label"], plot_dir / "packet_label_distribution.png",
                        "Packet label distribution", "packet_label")

    if "protocol" in flows.columns:
        plot_bar_counts(flows["protocol"], plot_dir / "flow_protocol_distribution.png",
                        "Flow protocol distribution", "protocol")

    if "flow_duration" in flows.columns:
        plot_hist_log1p(flows["flow_duration"], plot_dir / "flow_duration_log_hist.png",
                        "Flow duration distribution", "flow_duration")

    if "total_fwd_packets" in flows.columns and "total_backward_packets" in flows.columns:
        total_packets = (
            pd.to_numeric(flows["total_fwd_packets"], errors="coerce").fillna(0)
            + pd.to_numeric(flows["total_backward_packets"], errors="coerce").fillna(0)
        )
        plot_hist_log1p(total_packets, plot_dir / "flow_total_packets_log_hist.png",
                        "Total packets per flow", "total_packets")

    if "flow_iat_us" in packets.columns:
        plot_hist_log1p(packets["flow_iat_us"], plot_dir / "packet_flow_iat_log_hist.png",
                        "Packet flow IAT distribution", "flow_iat_us")

    if "direction_iat_us" in packets.columns:
        plot_hist_log1p(packets["direction_iat_us"], plot_dir / "packet_direction_iat_log_hist.png",
                        "Packet direction IAT distribution", "direction_iat_us")

    if "payload_len" in packets.columns:
        plot_hist_log1p(packets["payload_len"], plot_dir / "packet_payload_len_log_hist.png",
                        "Packet payload length distribution", "payload_len")



In [7]:
# -----------------------------
# Main
# -----------------------------

 # 统一转换为 Path 对象
out = Path("stage1_analysis")
plot_dir = ensure_output_dirs(out)
packets = Path("dataset-old/firewall_test_20250731_160700-stage1_packets.csv")
flows = Path("dataset-old/firewall_test_20250731_160700-stage1_flows.csv")
packets = read_csv_safely(packets, "packets")
flows = read_csv_safely(flows, "flows")

# Convert numeric columns while preserving obvious string columns.
packets = convert_numeric_columns(
    packets,
    exclude_cols={"record_type", "src_ip", "dst_ip", "source_ip", "destination_ip"}
)
flows = convert_numeric_columns(
    flows,
    exclude_cols={"record_type", "src_ip", "dst_ip", "source_ip", "destination_ip"}
)

# Save basic data quality reports.
save_missing_report(packets, out / "missing_values_packets.csv")
save_missing_report(flows, out / "missing_values_flows.csv")

save_numeric_summary(packets, out / "numeric_summary_packets.csv")
save_numeric_summary(flows, out / "numeric_summary_flows.csv")

# Flow-packet consistency checks.
consistency, label_consistency = check_flow_packet_consistency(flows, packets, out)

# Add log-IAT and packet order.
enriched_packets = create_enriched_packets(packets, out)

# Create flow-level ML-ready features.
flow_ml = create_flow_ml_features(flows, out)

# Feature-label correlation.
corrs = compute_label_correlations(flow_ml, out)

# Plots.
generate_plots(packets, flows, plot_dir)

# Markdown report.
write_markdown_report(
    packets=packets,
    flows=flows,
    enriched_packets=enriched_packets,
    flow_ml=flow_ml,
    consistency=consistency,
    label_consistency=label_consistency,
    corrs=corrs,
    out_dir=out,
)

print("\n[DONE]")
print(f"Analysis output directory: {out}")



[OK] Loaded packets: dataset\firewall_test_20250731_160700-stage1_packets.csv
     shape = (1003219, 29)
[OK] Loaded flows: dataset\firewall_test_20250731_160700-stage1_flows.csv
     shape = (38671, 73)
[OK] Wrote report: stage1_analysis\report.md

[DONE]
Analysis output directory: stage1_analysis


In [8]:
# 假设 flow_id 是数值类型（int/float）
count_zero = (packets['flow_id'] == 0).sum()
print(f"flow_id 为 0 的记录数：{count_zero}")

count_zero_p = (packets['packet_id'] == 0).sum()
print(f"packet_id 为 0 的记录数：{count_zero_p}")

flow_id 为 0 的记录数：0
packet_id 为 0 的记录数：11457


In [9]:
# 假设两列都是数值类型（int/float）
count_both_zero = ((packets['flow_id'] == 0) & (packets['packet_id'] == 0)).sum()
print(f"flow_id 和 packet_id 同时为 0 的记录数：{count_both_zero}")

# 假设两列都是数值类型（int/float）
count_or_zero = ((packets['flow_id'] == 0) | (packets['packet_id'] == 0)).sum()
print(f"flow_id 和 packet_id 有任意一个为 0 的记录数：{count_or_zero}")

flow_id 和 packet_id 同时为 0 的记录数：0
flow_id 和 packet_id 有任意一个为 0 的记录数：11457


In [10]:
# 假设两列都是数值类型（int/float）
count_fp_zero = ((packets['flow_id'] == 0) & (packets['packet_label'] == 1)).sum()
print(f"flow_id = 0 和 packet_label = 1 的记录数：{count_fp_zero}")

count_pp_zero = ((packets['packet_id'] == 0) & (packets['packet_label'] == 1)).sum()
print(f"packet_id = 0 和 packet_label = 1 的记录数：{count_pp_zero}")

flow_id = 0 和 packet_label = 1 的记录数：0
packet_id = 0 和 packet_label = 1 的记录数：51


In [11]:
count_all_zero = (((packets['flow_id'] == 0) | (packets['packet_id'] == 0)) & (packets['packet_label'] == 1)).sum()
print(f"记录数：{count_all_zero}")


记录数：51


In [12]:
count_both_zero = ((packets['flow_id'] == 0) & (packets['packet_id'] != 0)).sum()
print(f"flow_id == 0 和 packet_id != 0 的记录数：{count_both_zero}")

flow_id == 0 和 packet_id != 0 的记录数：0


In [13]:
filtered = packets[(packets['flow_id'] == 0) & (packets['packet_id'] != 0)]
filtered.to_csv('flow0_packetid_nonzero.csv', index=False)
print(f"保存了 {len(filtered)} 条记录到 flow0_packetid_nonzero.csv")

保存了 0 条记录到 flow0_packetid_nonzero.csv


In [14]:
filtered = packets[(packets['flow_id'] != 0) & (packets['packet_id'] == 0)]
filtered.to_csv('flow_nozero_packetid0.csv', index=False)
print(f"保存了 {len(filtered)} 条记录到 flow_nozero_packetid0.csv")

保存了 11457 条记录到 flow_nozero_packetid0.csv


In [15]:
import pandas as pd
from pathlib import Path

# 假设 packets 是已经加载好的 DataFrame
# 如果还没有加载，请先读取：
# packets = pd.read_csv('your_packets_file.csv')

# 步骤1：找出满足 “flow_id != 0 且存在 packet_id == 0 且 packet_label == 1” 的所有 flow_id
flow_ids_with_condition = packets[
    (packets['flow_id'] != 0) &
    (packets['packet_id'] == 0) &
    (packets['packet_label'] == 1)
]['flow_id'].unique()

print(f"符合条件的 flow_id 数量: {len(flow_ids_with_condition)}")

# 步骤2：对于这些 flow_id，提取所有 packet_label == 1 的记录（无论 packet_id 是否为 0）
result = packets[
    (packets['flow_id'].isin(flow_ids_with_condition)) &
    (packets['packet_label'] == 1)
]

# 步骤3：按 flow_id 和 packet_id 排序以便查看
result = result.sort_values(['flow_id', 'packet_id'])

# 分组统计每个 flow 中 packet_label == 1 的 packet_id 情况
stats = result.groupby('flow_id').agg(
    总label1包数=('packet_id', 'count'),
    是否有packet_id0=('packet_id', lambda x: (x == 0).any()),
    是否有其他packet_id=('packet_id', lambda x: ((x != 0) & (x.notna())).any())
).reset_index()
print(stats)

# 显示结果前几行
print(f"总共找到 {len(result)} 条 packet_label == 1 的记录，涉及 {len(flow_ids_with_condition)} 个流量。")
print(result.head())

# 可选：保存到 CSV
out_dir = Path("stage1_analysis")
out_dir.mkdir(exist_ok=True)
result.to_csv(out_dir / "flows_with_packet0_label1_all_label1_packets.csv", index=False)
print(f"结果已保存至 {out_dir / 'flows_with_packet0_label1_all_label1_packets.csv'}")

符合条件的 flow_id 数量: 40
             flow_id  总label1包数  是否有packet_id0  是否有其他packet_id
0    586959886115832          2           True            True
1    591147588971802          1           True           False
2    638367098695589          1           True           False
3    731039072620777          4           True           False
4    790381537562621          1           True           False
5    910633175876293          1           True           False
6    980832669095548          1           True           False
7    984670365785082          1           True           False
8    997952354027343          1           True           False
9   1014880779266473          2           True            True
10  1020710785359680          2           True            True
11  1029073001108768          9           True           False
12  1039835520841558          1           True           False
13  1191007534288165          1           True           False
14  1250939516981880          1   

In [16]:
import pandas as pd

# 假设 packets 是你的 DataFrame
# 首先筛选出满足条件的 flow_id（flow_id != 0, packet_id == 0, packet_label == 1）
condition = (packets['flow_id'] != 0) & (packets['packet_id'] == 0) & (packets['packet_label'] == 1)
target_flows = packets.loc[condition, 'flow_id'].unique()  # 去重的 flow_id

print(f"符合条件的 flow_id 数量: {len(target_flows)}")
# print(f"符合条件的 flow_ids: {target_flows}")

# fid = 591147588971802
# cur = packets[packets['flow_id'] == fid][['flow_id', 'packet_id', 'packet_label']]
# print(cur)
#
# # 统计 packet_id == 0 & packet_label == 1 的记录数
# count_p0_l1 = ((cur['packet_id'] == 0) & (cur['packet_label'] == 1)).sum()
#
# # 统计 packet_id != 0 & packet_label == 1 的记录数
# count_pn0_l1 = ((cur['packet_id'] != 0) & (cur['packet_label'] == 1)).sum()
#
# print(f"Flow {fid}: packet_id==0&label==1 -> {count_p0_l1} 条, packet_id!=0&label==1 -> {count_pn0_l1} 条")
#
# 循环遍历每个目标 flow_id
for fid in target_flows:
    # 筛选当前 flow 的数据
    cur = packets[packets['flow_id'] == fid][['flow_id', 'packet_id', 'packet_label']]
    print(cur)

    # 统计 packet_id == 0 & packet_label == 1 的记录数
    count_p0_l1 = ((cur['packet_id'] == 0) & (cur['packet_label'] == 1)).sum()

    # 统计 packet_id != 0 & packet_label == 1 的记录数
    count_pn0_l1 = ((cur['packet_id'] != 0) & (cur['packet_label'] == 1)).sum()

    print(f"Flow {fid}: packet_id==0&label==1 -> {count_p0_l1} 条, packet_id!=0&label==1 -> {count_pn0_l1} 条")

#
# # 可选：按 flow_id 分组查看每个 flow 的详细统计
# grouped = packets[packets['flow_id'].isin(target_flows)].groupby('flow_id')
# stats = grouped.apply(
#     lambda g: pd.Series({
#         'packet_id==0 & label==1': ((g['packet_id'] == 0) & (g['packet_label'] == 1)).sum(),
#         'packet_id!=0 & label==1': ((g['packet_id'] != 0) & (g['packet_label'] == 1)).sum()
#     })
# ).reset_index()
#
# print("\n每个 flow_id 的详细统计：")
# print(stats)

符合条件的 flow_id 数量: 40
                 flow_id  packet_id  packet_label
5314     591147588971802       5304             0
5871     591147588971802       5845             0
6261     591147588971802       6234             0
9065     591147588971802       9023             0
9586     591147588971802       9541             0
10241    591147588971802          0             0
10242    591147588971802          0             1
10243    591147588971802      10195             0
11014    591147588971802      10964             0
11015    591147588971802      10965             0
13657    591147588971802      13592             0
20489    591147588971802      20400             0
21825    591147588971802      21731             0
21826    591147588971802      21732             0
22157    591147588971802      22062             0
22166    591147588971802      22071             0
22651    591147588971802      22556             0
1011189  591147588971802          0             0
1011190  591147588971802     

In [18]:
condition = packets['ttl_or_hop_limit'] == -1
c2 = (packets['ttl_or_hop_limit'] == -1) & (packets['flow_id'] == 0)
print(f"符合条件的数量: {len(condition)}")
print(f"符合条件的数量: {len(c2)}")

符合条件的数量: 1003219
符合条件的数量: 1003219
